# Campus Café Detective — Worked Example & Discussion

This worked example revisits the guided coffee comparison after your own attempt. It is a learner-facing explanation, not private assessment guidance. The independent investigation has multiple valid answers; use its stated evidence checks to assess yours.

Run this notebook from the beginning. It contains the same setup, five CSV inputs and unchanged request helper as the project notebook. Offline reports are author-written teaching examples, not live model outputs. The following commentary explains the choices behind the calculations and one prompt revision.

## 1. Prepare your workspace

Keep `sales.csv`, `weather.csv`, `academic_calendar.csv`, `campus_events.csv` and `promotions.csv` beside this notebook. Run the cells in order in a fresh kernel. The offline route needs Pandas and Matplotlib; it makes no model requests. The browser workspace provides these packages. Locally, install the companion package requirements and start JupyterLab as explained in Section 2.3.

For optional live responses, keep your own `.env` at the extracted package root beside `COMPANION_VERSION`, with `OPENAI_API_KEY` and `OPENAI_MODEL`. A standalone copy can keep `.env` beside the notebook. Never put a key in a code cell. Live requests need a standard local Python kernel; keep offline mode in the browser.

The next cell repeats the **unchanged** `get_completion(system_prompt, user_prompt)` helper from the preceding experiments so that this notebook works independently. It also retains the existing offline/live switch and supplied-response registration. Leave the supplied helper unchanged; focus on the data and prompts below.

In [ ]:
# Shared offline/live request adapter. No API key is stored in this notebook.
# Code origin: get_completion(system_prompt, user_prompt) was introduced in Section 2.3.
# Notebook extension used from Section 3.2: supplied/live mode and interaction records.
# Evaluation notebooks reuse this helper without changing its two-argument interface.
import json
import os
import sys

MODE = "offline"
MODEL = ""
_client = None
request_records = []
OFFLINE_SAMPLES = {}

# Match supplied examples only when both messages are unchanged.
def _request_key(system_prompt, user_prompt):
    return json.dumps([system_prompt.strip(), user_prompt.strip()], ensure_ascii=False)

# Register a supplied teaching response; this does not contact OpenAI.
def register_sample(system_prompt, user_prompt, response):
    OFFLINE_SAMPLES[_request_key(system_prompt, user_prompt)] = response

# Locate one package-root configuration, while retaining standalone notebook use.
def _configuration_file():
    from pathlib import Path
    bases = [Path.cwd()]
    if "__file__" in globals():
        bases.insert(0, Path(__file__).resolve().parent)
    for base in bases:
        for folder in [base, *base.parents]:
            if (folder / "COMPANION_VERSION").is_file():
                return folder / ".env"
    # Standalone notebooks keep their configuration beside the notebook.
    return bases[0] / ".env"

# Load the private configuration and prepare an OpenAI client when live mode is chosen.
def configure_live(model=""):
    global MODE, MODEL, _client
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("The OpenAI SDK needs a standard Python kernel. Keep offline mode in browser Python.")
    try:
        from openai import OpenAI
        from dotenv import load_dotenv
    except ImportError:
        raise RuntimeError("Install openai and python-dotenv in your Python kernel, then rerun setup.") from None
    # Read the key and model from the local file without printing the key.
    load_dotenv(_configuration_file())
    selected_model = model.strip() or os.getenv("OPENAI_MODEL", "").strip()
    if not selected_model or selected_model.startswith("YOUR_"):
        raise ValueError("Set OPENAI_MODEL in your .env file before choosing live mode.")
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if not key or key.startswith("YOUR_"):
        raise ValueError("Copy your own OpenAI key into the .env file before choosing live mode.")
    _client = OpenAI(timeout=45.0, max_retries=0)
    MODEL = selected_model
    MODE = "live"
    print("OpenAI configured from .env. No key is printed.")


def get_completion(system_prompt, user_prompt):
    if MODE == "live":
        if _client is None:
            raise RuntimeError("Run configure_live(MODEL) in a private Python kernel first.")
        try:
            # Send role instructions separately from the user task and data.
            response = _client.responses.create(model=MODEL, instructions=system_prompt, input=user_prompt, max_output_tokens=1000, store=False)
        except Exception:
            raise RuntimeError("API request failed. Check your model, account access and connection; no key has been printed.") from None
        text = response.output_text
        print("[Live OpenAI response]")
    else:
        # Offline mode returns stored text, not a newly generated model answer.
        text = OFFLINE_SAMPLES.get(_request_key(system_prompt, user_prompt))
        if text is None:
            print("[No saved response for these edited instructions]")
            return "No model was called. Use OpenAI mode in a standard Python kernel, or label a response you write yourself."
        print("[Supplied teaching response — no API call]")
    # Keep messages and responses for evaluation; never record the API key.
    request_records.append({"mode": MODE, "model": MODEL if MODE == "live" else "supplied-example", "system_prompt": system_prompt, "user_prompt": user_prompt, "response": text})
    return text

# Release the API client after finishing the experiment.
def clear_live():
    global _client, MODE
    if _client is not None:
        _client.close()
    _client = None
    MODE = "offline"
    print("Live client cleared from this kernel.")


In [ ]:
# Leave this False to run the complete guided example without API requests.
USE_OPENAI = False
if USE_OPENAI:
    configure_live()
else:
    print("Offline mode: Python runs normally; model responses are supplied examples.")


## 2. Inspect the five datasets

The fictional records cover 5 January–4 April 2026: 90 days and five products. `sales.csv` has one row per date and product. Weather and calendar have one row per day; events and promotions occur only on some dates.

**Predict:** would comparing total coffee sales be fair if the two groups contain different numbers of days? Write your answer before calculating.

`pd.read_csv()` loads a CSV into a table called a DataFrame. `parse_dates` reads dates as dates. `head()` shows the first rows; it does not change the data. The following dictionary simply keeps the five tables together.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Read files from the notebook's folder. No network or external API is used.
sales = pd.read_csv("sales.csv", parse_dates=["date"])
weather = pd.read_csv("weather.csv", parse_dates=["date"])
calendar = pd.read_csv("academic_calendar.csv", parse_dates=["date"])
events = pd.read_csv("campus_events.csv", parse_dates=["date"])
promotions = pd.read_csv("promotions.csv", parse_dates=["date"])

tables = {"Sales": sales, "Weather": weather, "Calendar": calendar,
          "Events": events, "Promotions": promotions}
for name, table in tables.items():
    print(f"\n{name} — {len(table)} rows")
    print(table.head(3).to_string(index=False))
print("\nDate range:", sales["date"].min().date(), "to", sales["date"].max().date())


**Read the columns before joining.** `quantity_sold` counts units, `unit_price` is the actual price in euros after any discount, and `revenue` is quantity × actual price. Do not apply the discount again. `is_lecture_day` marks the fictional calendar's non-holiday weekday operating pattern; it can overlap an examination period. Expected event attendance is not a count of café customers.

`merge()` brings columns together using matching keys. Join daily context on `date`, but promotions on **both** `date` and `product_id`. Otherwise, one product's discount could be assigned to another product. `how="left"` retains sales days without an event or promotion. There is at most one event per date in these supplied files.

In [ ]:
# Attach context without adding or removing sales rows.
combined = sales.merge(weather, on="date", validate="many_to_one")
combined = combined.merge(calendar, on="date", validate="many_to_one")
combined = combined.merge(
    events[["date", "event_name", "expected_attendance"]],
    on="date", how="left", validate="many_to_one"
)
combined = combined.merge(
    promotions[["date", "product_id", "discount_percent"]],
    on=["date", "product_id"], how="left", validate="one_to_one"
)

# In these generated files, no matching record means no recorded event/promotion.
combined["event_name"] = combined["event_name"].fillna("No recorded event")
combined["expected_attendance"] = combined["expected_attendance"].fillna(0)
combined["discount_percent"] = combined["discount_percent"].fillna(0)
print("Sales rows before and after joining:", len(sales), len(combined))
print("Revenue unchanged:", round(sales["revenue"].sum(), 2) ==
      round(combined["revenue"].sum(), 2))


## 3. Calculate and visualise the comparison

Our guided question is: **How do daily coffee sales differ between lecture days and other days? What can these records establish?**

Filtering selects coffee rows. `groupby()` separates the two calendar groups; `agg()` calculates their day counts, total units and mean daily units. The mean allows a per-day comparison when group sizes differ. This is a descriptive comparison, not proof that lectures caused a difference.

In [ ]:
# Select one product; each remaining row represents one day.
coffee = combined[combined["product_name"] == "Coffee"].copy()
summary = coffee.groupby("is_lecture_day")["quantity_sold"].agg(
    days="count", total_units="sum", mean_daily_units="mean"
).round(2)
summary.index = ["Lecture day" if value else "Other day" for value in summary.index]
print(summary.to_string())

lecture_mean = summary.loc["Lecture day", "mean_daily_units"]
other_mean = summary.loc["Other day", "mean_daily_units"]
difference = round(lecture_mean - other_mean, 2)
print(f"\nDifference in the displayed daily means: {difference:.2f} coffees per day")


In [ ]:
# Show both the overall comparison and variation across individual days.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
summary["mean_daily_units"].plot.bar(ax=axes[0], color=["#d69710", "#174283"])
axes[0].set(title="Coffee: mean daily sales", ylabel="Coffees per day")
axes[0].tick_params(axis="x", rotation=0)
coffee.plot(x="date", y="quantity_sold", ax=axes[1], legend=False, color="#174283")
axes[1].set(title="Coffee: daily variation", xlabel="Date", ylabel="Coffees sold")
fig.tight_layout()
plt.show()


**Observe and explain:** which group has the higher mean? Do the daily sales fluctuate? Why do unequal day counts matter? A higher mean does not mean that every day in that group had higher sales.

Before prompting, inspect a small selection of the contextual columns. We inspected all five files, but the guided comparison uses the calendar groups. Weather, events and promotions are possible context for further investigation; their presence does not prove a cause.

In [ ]:
columns = ["date", "quantity_sold", "is_lecture_day", "temperature_c",
           "event_name", "expected_attendance", "discount_percent"]
print(coffee[columns].head(6).to_string(index=False))

# Package only the calculated evidence needed for this question.
evidence = (
    "Synthetic café data, 5 January–4 April 2026. Product: Coffee.\n"
    + summary.to_string()
    + f"\nDifference of displayed means (lecture minus other): {difference:.2f}."
    + "\nThe calendar groups have different day counts. These are observational comparisons."
)
print("\nEvidence to include in the prompt:\n" + evidence)


## 4. Compare basic and CoT prompts

Keep the system prompt, question, evidence and model fixed. Only the user instruction changes. Do not send all five files to the model simply because they are available.

**Predict:** will requesting steps change the numbers, the presentation or both? What would count as an unsupported statement?

In [ ]:
system_prompt = "You are a café data analyst. Use only the supplied evidence."
question = "How do daily coffee sales differ between lecture days and other days?"
basic_instruction = "Answer the question briefly."
cot_instruction = "Give a short step-by-step comparison, followed by a conclusion."

# Keep the shared question and evidence visible, separate from each instruction.
basic_prompt = question + "\n" + evidence + "\n" + basic_instruction
cot_prompt = question + "\n" + evidence + "\n" + cot_instruction
print("BASIC PROMPT\n" + basic_prompt)
print("\nCOT PROMPT\n" + cot_prompt)


### Offline examples and optional live responses

The next cell registers **author-written teaching responses**, using the displayed Python figures. They are not recorded OpenAI outputs and do not demonstrate how a particular model will perform. They let you practise reading and checking reports without a key.

In live mode the same `get_completion()` calls request fresh responses instead. This stage makes **two API requests**; the revision later makes one more. Read each result before rerunning. A real response may already include everything requested by the later revision.

In [ ]:
# Supplied responses illustrate a comparison; they are not generated by an LLM.
supplied_basic = (
    f"Coffee sales averaged {lecture_mean:.2f} on lecture days and "
    f"{other_mean:.2f} on other days. The difference was {difference:.2f} per day."
)
supplied_cot = (
    f"1. Compare mean daily sales, rather than totals across unequal day counts.\n"
    f"2. Lecture days: {lecture_mean:.2f}; other days: {other_mean:.2f} coffees per day.\n"
    f"3. Subtract the displayed means: {lecture_mean:.2f} - {other_mean:.2f} = {difference:.2f}.\n"
    "Conclusion: the lecture-day group had higher average daily coffee sales."
)
register_sample(system_prompt, basic_prompt, supplied_basic)
register_sample(system_prompt, cot_prompt, supplied_cot)

basic_response = get_completion(system_prompt, basic_prompt)
cot_response = get_completion(system_prompt, cot_prompt)
print("\nBASIC RESPONSE\n" + basic_response)
print("\nCOT RESPONSE\n" + cot_response)


## 5. Refine one instruction

Inspect the actual response first. Does it distinguish an observed difference from a causal explanation? Does it say what information is still needed? The supplied CoT example correctly reports the comparison but leaves these limitations unstated.

For the guided revision, keep the question and evidence unchanged and add one requirement: explicitly separate the observed result from what cannot be concluded. If your live response already does this well, explain that; do not invent a weakness just to justify a revision.

In [ ]:
revised_instruction = (
    cot_instruction
    + " Clearly separate the observed difference from causal claims, "
      "and state one limitation or additional comparison needed."
)
revised_prompt = question + "\n" + evidence + "\n" + revised_instruction
supplied_revised = (
    supplied_cot
    + "\nLimitation: these records show an association, not that lectures caused the difference. "
      "Weekday/weekend patterns, weather, promotions or events may overlap. "
      "Compare more similar days before proposing a cause."
)
register_sample(system_prompt, revised_prompt, supplied_revised)
revised_response = get_completion(system_prompt, revised_prompt)
print("\nREVISED RESPONSE\n" + revised_response)


## 6. Check the claims and explain the change

Use the original coffee rows to verify the group means separately. The following calculation uses sums and day counts instead of repeating the aggregation above. It checks the reference figures, **not automatically the wording of the model's response**. Read each response and compare its numbers and claims with these results.

In [ ]:
lecture_values = coffee.loc[coffee["is_lecture_day"], "quantity_sold"]
other_values = coffee.loc[~coffee["is_lecture_day"], "quantity_sold"]
checked_lecture = round(lecture_values.sum() / len(lecture_values), 2)
checked_other = round(other_values.sum() / len(other_values), 2)
print(f"Lecture: {lecture_values.sum()} units / {len(lecture_values)} days = {checked_lecture:.2f}")
print(f"Other:   {other_values.sum()} units / {len(other_values)} days = {checked_other:.2f}")
print(f"Difference of displayed means: {checked_lecture - checked_other:.2f}")
print("\nNow check every numerical claim in the three responses against these figures.")


Write your observations here:

- Did each response report the correct figures and units?
- Did the CoT response show useful intermediate steps?
- What did the revision improve, if anything? Did it introduce a new error?
- Which statements are supported observations, and which would require more evidence?

The offline examples deliberately illustrate a specific omission and revision. They are not evidence that CoT or a longer prompt is generally better. Live responses can differ.

## 7. Investigate your own question

Choose a question different from the guided lecture-day coffee comparison. For example, compare tea sales on cold versus other days, or sandwich sales on recorded event days versus days without a recorded event. Define the groups clearly and explain why the comparison is useful.

1. Write your question and prediction before calculating.
2. Select the product and relevant context from `combined`.
3. Calculate group sizes and an appropriate per-day measure; make one chart.
4. Prepare a concise evidence table. Check its figures before prompting.
5. Write basic and CoT prompts for that same evidence. If live mode is available, run them, identify a weakness and make one targeted revision.
6. Without live access, write two **reader-written draft reports**, one answer-only and one with short solution steps. Improve one weakness and label these as your own writing. Do not present them as model outputs; the helper has no stored response for a new task.
7. Verify the report against your Python results. Explain one limitation and what information you would need next.

The independent task is yours to implement. You can reuse the simple filtering, grouping, plotting and prompt-construction patterns above; no new API, tool framework or model-training code is needed.

In [ ]:
# Your independent analysis goes here.
# Start by writing your own question, then select relevant rows from combined.
# Add cells as needed; keep calculations and prompts separate.
independent_question = ""
print("Write your independent question and analysis before consulting the walkthrough.")


## 8. Write your report and reflect

Use this compact structure:

- **Question:** what did you investigate?
- **Findings:** the checked figures, group sizes and units, supported by your chart.
- **Prompt comparison:** what changed between responses and after your revision?
- **Limitations:** what does the comparison not establish?
- **Next step:** a reasonable additional comparison or information request.

Label supplied responses, live responses and your own writing. A successful project includes an independent investigation, not only a run of the guided cells. Check that the joins retained all 450 sales rows and that you did not subtract a discount twice. Save your notebook and any chart you want to retain. Review the book's Solution & Discussion afterwards.

## Discussion of the worked example

Daily means are preferable to raw totals for these differently sized groups. Joining context preserves sales rows; a missing event record is not a missing sales day. The chart shows variation that an overall mean hides. Weather, events and promotions are inspected as context, but the prompt's evidence supports only the descriptive calendar comparison.

The basic supplied report gives the figures. The supplied CoT report makes the subtraction explicit, but does not explain the limits of that comparison. The revised report adds that limit without changing the data or recalculating the result. This teaches purposeful revision; it does not establish that asking for more steps improves every model response.

For your independent question, check the product, grouping rule, group sizes, arithmetic and units. A different question need not reproduce these coffee figures. Claims about why customers bought more require additional evidence; planned event attendance cannot establish actual café visits.

## An independent example: tea sales and cold days

Here is one completed transfer example. Our new question is: **How do mean daily tea sales compare on days below 5°C and other days?** We define the threshold before comparing, keep one row per day, and compare averages rather than unequal totals. This is a new question using the same Pandas patterns, not a new prompting method.

The calculation below is reproducible. The report is explicitly **author-written**, not a new model response. You can use this checked table in your own basic and CoT requests if you choose live mode; this walkthrough does not automatically make additional requests.

In [ ]:
# Select Tea, then define the two weather groups explicitly.
tea = combined[combined["product_name"] == "Tea"].copy()
tea["cold_day"] = tea["temperature_c"] < 5
tea_summary = tea.groupby("cold_day")["quantity_sold"].agg(
    days="count", total_units="sum", mean_daily_units="mean"
).round(2)
tea_summary.index = ["Below 5°C" if value else "5°C or warmer" for value in tea_summary.index]
print(tea_summary.to_string())
ax = tea_summary["mean_daily_units"].plot.bar(
    color=["#174283", "#d69710"], title="Tea: mean daily sales", ylabel="Teas per day", rot=0
)
plt.tight_layout()
plt.show()

# Print separate prompt elements; no additional model request is made.
tea_question = "How do daily tea sales compare below 5°C and at 5°C or warmer?"
tea_evidence = tea_summary.to_string()
tea_cot_prompt = tea_question + "\n" + tea_evidence + "\n" + cot_instruction
tea_revision = tea_question + "\n" + tea_evidence + "\n" + revised_instruction
print("\nCOT PROMPT\n" + tea_cot_prompt)
print("\nREVISED PROMPT\n" + tea_revision)

cold_mean = tea_summary.loc["Below 5°C", "mean_daily_units"]
warmer_mean = tea_summary.loc["5°C or warmer", "mean_daily_units"]
print("\nAUTHOR-WRITTEN REPORT — NOT A MODEL OUTPUT")
print(f"Tea averaged {cold_mean:.2f} units per day below 5°C and {warmer_mean:.2f} at 5°C or warmer.")
print(f"The difference between displayed means is {cold_mean - warmer_mean:.2f} teas per day.")
print("These are observed group averages; they do not prove a temperature effect.")
print("Next compare similar calendar days and inspect promotions or events before proposing a cause.")


**Check and discuss:** recompute each mean from the selected rows and count the days in each group. The threshold, units and direction of subtraction must be clear. The revision adds a limitation requirement while leaving the tea evidence unchanged. If you ran your own prompts, compare the actual responses; the author-written report is a reference for evidence and interpretation, not an expected verbatim model answer.

In [ ]:
# Close an optional live client after all requested work is finished.
clear_live()
